In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
%cd /content/drive/MyDrive/Foodalyze

/content/drive/MyDrive/Foodalyze


In [13]:
!pip install dvc[all] ultralytics mlflow

In [14]:
!git clone https://github.com/alina1114/Foodalyze.git
%cd Foodalyze

Cloning into 'Foodalyze'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 23 (delta 2), reused 18 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 6.47 KiB | 828.00 KiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/drive/MyDrive/Foodalyze/Foodalyze


In [15]:
DATA_PATH = "/content/drive/MyDrive/Foodalyze/data/IndianFoodDatasetFinalFiltered"


In [ ]:
# pragma: allowlist secret

In [ ]:
!pip uninstall -y wandb
!pip install --upgrade ultralytics


In [ ]:
import os

# Root directory of dataset
root = "/content/drive/MyDrive/Foodalyze/data/IndianFoodDatasetFinalFiltered"

# Define your class names
classes = [
    "aloo_gobi","aloo_matar","aloo_methi","aloo_shimla_mirch","aloo_tikki","bhatura",
    "bhindi_masala",
    "biryani",
    "boondi",
    "butter_chicken",
    "chak_hao_kheer",
    "cham_cham",
    "chana_masala",
    "chapati",
    "chhena_kheeri",
    "chicken_razala",
    "chicken_tikka",
    "chicken_tikka_masala",
    "chikki",
    "daal_puri",
    "dal_makhani",
    "dal_tadka",
    "dharwad_pedha",
    "double_ka_meetha",
    "dum_aloo",
    "gajar_ka_halwa",
    "gulab_jamun",
    "imarti",
    "jalebi",
    "kachori",
    "kadai_paneer",
    "kajjikaya",
    "kalakand",
    "karela_bharta",
    "kofta",
    "lassi",
    "makki_di_roti_sarson_da_saag",
    "malapua",
    "misi_roti",
    "misti_doi",
    "mysore_pak",
    "naan",
    "navrattan_korma",
    "palak_paneer",
    "paneer_butter_masala",
    "poha",
    "poornalu",
    "pootharekulu",
    "qubani_ka_meetha",
    "rabri",
    "ras_malai",
    "rasgulla",
    "sandesh",
    "sheer_korma",
    "sheera",
    "sohan_halwa",
    "sohan_papdi",
    "unni_appam"
]

# Create mapping: name -> index
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}

# Go through all label files
for split in ["train", "valid"]:
    labels_dir = os.path.join(root, split, "labels")
    for file in os.listdir(labels_dir):
        path = os.path.join(labels_dir, file)
        lines = open(path).read().strip().splitlines()
        new_lines = []
        for line in lines:
            parts = line.split()
            if parts[0] in class_to_idx:
                cls_id = class_to_idx[parts[0]]
                new_line = " ".join([str(cls_id)] + parts[1:])
                new_lines.append(new_line)
            else:
                print(f"Unknown label name: {parts[0]} in {file}")
        with open(path, "w") as f:
            f.write("\n".join(new_lines))
print("Conversion complete! All class names replaced with numeric IDs.")


In [ ]:
import sys
import os
import types
import builtins

# --- Fake the wandb module BEFORE YOLO imports it ---
fake_wandb = types.SimpleNamespace(
    init=lambda *a, **k: None,
    log=lambda *a, **k: None,
    finish=lambda *a, **k: None,
    define_metric=lambda *a, **k: None,
    config={},
    run=None,
    Table=lambda *a, **k: None
)
sys.modules["wandb"] = fake_wandb

# Also block any re-import attempts
builtins.__import__ = (lambda orig_import:
    (lambda name, *args, **kwargs: fake_wandb if name == "wandb" else orig_import(name, *args, **kwargs))
)(__import__)

print("✅ Completely disabled wandb imports system-wide.")


In [6]:
!nohup mlflow server \
  --backend-store-uri runs/mlflow \
  --default-artifact-root runs/mlflow/artifacts \
  --host 127.0.0.1 \
  --port 5000 > mlflow.log 2>&1 &


In [ ]:
!curl http://127.0.0.1:5000


In [8]:
import os
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "Foodalyze"

In [ ]:
# pragma: allowlist secret
from ultralytics import YOLO
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Foodalyze")

with mlflow.start_run():
    model = YOLO("yolov8n.pt")
    results = model.train(
        data="/content/drive/MyDrive/Foodalyze/data/IndianFoodDatasetFinalFiltered/data.yaml",
        epochs=30,
        imgsz=640,
        name="foodalyze_yolov8_colab"
    )

    mlflow.log_param("epochs", 30)
    mlflow.log_param("imgsz", 640)
    mlflow.log_artifact("/content/runs/detect/foodalyze_yolov8_colab/weights/best.pt")
